In [4]:
"""
Feature selection with the Mann-Whitney U test.

- Loads the wPLI feature CSV
- Finds the label and patient id columns (case-insensitive, so
  'Schizophrenia_Label' and 'schizophrenia_label' both work)
- Runs a two-sided Mann-Whitney U test on every feature
- Keeps features with p < 0.05
- Saves reduced_features_dataset.csv (patient id + label + significant features)
"""

import sys
import pandas as pd
from scipy.stats import mannwhitneyu

# ----------------------------- Settings ------------------------------
INPUT_FILE = "wpli_features (Copy).csv"   # change the path if needed
OUTPUT_FILE = "reduced_features_dataset.csv"
LABEL_NAME = "schizophrenia_label"
ID_NAME = "patient_id"                    # also matches "patient id", "Patient_ID"
ALPHA = 0.05


def find_column(df, wanted):
    """Find a column ignoring case, spaces, and underscores."""
    clean = lambda s: str(s).strip().lower().replace(" ", "").replace("_", "")
    for col in df.columns:
        if clean(col) == clean(wanted):
            return col
    raise KeyError(
        f"Could not find a column like '{wanted}'. "
        f"Columns found: {list(df.columns[:10])} ..."
    )


def main():
    # 1. Load and inspect
    df = pd.read_csv(INPUT_FILE)
    print(f"Loaded '{INPUT_FILE}' with shape {df.shape}")

    label_col = find_column(df, LABEL_NAME)
    id_col = find_column(df, ID_NAME)
    print(f"Label column: '{label_col}' | ID column: '{id_col}'")

    unique_labels = df[label_col].dropna().unique().tolist()
    print(f"Exact unique values in '{label_col}': {[repr(v) for v in unique_labels]}")
    print(df[label_col].value_counts(dropna=False).to_string())

    if len(unique_labels) != 2:
        sys.exit(f"ERROR: expected 2 classes but found {len(unique_labels)}.")

    # 2. Dynamic grouping (sorted so the order is the same every run)
    group1_value, group2_value = sorted(unique_labels, key=str)
    print(f"\nGroup 1 = {group1_value!r} | Group 2 = {group2_value!r}")

    # 3. Type handling: features = everything except metadata
    metadata_cols = [id_col, label_col]
    feature_cols = [c for c in df.columns if c not in metadata_cols]
    features = df[feature_cols].apply(pd.to_numeric, errors="coerce")

    # Drop columns that are entirely NaN (nothing to test)
    all_nan = features.columns[features.isna().all()].tolist()
    if all_nan:
        print(f"Dropping {len(all_nan)} feature(s) with no numeric values.")
        features = features.drop(columns=all_nan)

    n_original = features.shape[1]

    # 4. Statistical test
    mask1 = df[label_col] == group1_value
    mask2 = df[label_col] == group2_value

    significant = []
    skipped = 0
    for col in features.columns:
        g1 = features.loc[mask1, col].dropna()
        g2 = features.loc[mask2, col].dropna()

        if len(g1) == 0 or len(g2) == 0:
            skipped += 1
            continue
        try:
            _, p = mannwhitneyu(g1, g2, alternative="two-sided")
        except ValueError:
            # e.g. all values identical in both groups
            skipped += 1
            continue

        # 5. Filtering
        if p < ALPHA:
            significant.append(col)

    # 6. Reporting
    print(f"\nOriginal number of features:       {n_original}")
    print(f"Significant features (p < {ALPHA}): {len(significant)}")
    if skipped:
        print(f"Features skipped (empty group or constant values): {skipped}")

    # 7. Export
    reduced = pd.concat(
        [df[[id_col, label_col]], features[significant]], axis=1
    )
    reduced.to_csv(OUTPUT_FILE, index=False)
    print(f"Saved '{OUTPUT_FILE}' with shape {reduced.shape}")


if __name__ == "__main__":
    main()

Loaded 'wpli_features (Copy).csv' with shape (97, 857)
Label column: 'Schizophrenia_Label' | ID column: 'Patient_ID'
Exact unique values in 'Schizophrenia_Label': ['0', '1']
Schizophrenia_Label
0    50
1    47

Group 1 = 0 | Group 2 = 1

Original number of features:       855
Significant features (p < 0.05): 95
Saved 'reduced_features_dataset.csv' with shape (97, 97)


In [13]:
"""
Frequency-band analysis of wPLI features: Healthy vs Schizophrenia.

What this script does
---------------------
1. Loads the CSV and finds the patient-id and label columns.
2. Reads each feature name (e.g. wPLI_Gamma_T5-P3) to get the
   frequency band (Gamma) and the electrode pair (T5, P3).
3. Tests every feature (Mann-Whitney U, two-sided), and adds:
   - effect size (rank-biserial r: + means higher in the 2nd group)
   - FDR-corrected p-value (Benjamini-Hochberg)
4. Tests every band as a whole (average wPLI per person per band).
5. Saves tables (CSV) and figures (PNG) in an output folder.

Usage
-----
    python band_group_analysis.py
    python band_group_analysis.py --input my_file.csv --outdir results
"""

import argparse
import re
from pathlib import Path

import numpy as np

# --- COMPATIBILITY PATCH FOR NUMPY 2.0+ ---
# Older versions of Seaborn use np.product which was removed in NumPy 2.0.
# This aliases it to np.prod so Seaborn doesn't crash during the heatmap generation.
if not hasattr(np, "product"):
    np.product = np.prod
# ------------------------------------------

import pandas as pd
import matplotlib
matplotlib.use("Agg")                      # save figures without opening windows
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

# Fix for scipy < 1.11.0 missing false_discovery_control
try:
    from scipy.stats import false_discovery_control
except ImportError:
    def false_discovery_control(ps, method="bh"):
        ps = np.asarray(ps)
        order = np.argsort(ps)
        ps_sorted = ps[order]
        n = len(ps)
        q = np.zeros(n)
        prev_q = 1.0
        for i in range(n - 1, -1, -1):
            fdr = min(prev_q, (n / (i + 1)) * ps_sorted[i])
            q[i] = fdr
            prev_q = fdr
        res = np.zeros(n)
        res[order] = q
        return np.clip(res, 0, 1)

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ----------------------------- Settings ------------------------------
LABEL_NAME = "schizophrenia_label"
ID_NAME = "patient id"

# Updated to match the string values in your CSV
LABEL_NAMES = {"healthy": "Healthy", "schizophrenia": "Schizophrenia"}
BAND_ORDER = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
CHANNEL_ORDER = ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T3", "C3", "Cz",
                 "C4", "T4", "T5", "P3", "Pz", "P4", "T6", "O1", "O2"]
FEATURE_PATTERN = re.compile(r"^wPLI_([A-Za-z]+)_([A-Za-z0-9]+)-([A-Za-z0-9]+)$")
TOP_N_BOXPLOTS = 12
TOP_N_HEATMAP = 30

try:
    pd.get_option("mode.use_inf_as_na")
except Exception:
    try:
        from pandas._config import config as _pd_cfg
        _pd_cfg.register_option("mode.use_inf_as_na", False, "compat shim for older seaborn")
    except Exception:
        pass

sns.set_theme(style="whitegrid", context="notebook")

# ----------------------------- Helpers -------------------------------
def find_column(df, wanted):
    clean = lambda s: str(s).strip().lower().replace(" ", "").replace("_", "")
    for col in df.columns:
        if clean(col) == clean(wanted):
            return col
    raise KeyError(f"Could not find a column like '{wanted}'.")

def sig_stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

def run_mwu(a, b):
    a, b = pd.Series(a).dropna(), pd.Series(b).dropna()
    if len(a) == 0 or len(b) == 0:
        return None
    try:
        u, p = mannwhitneyu(a, b, alternative="two-sided")
    except ValueError:
        return None
    return p, 2 * u / (len(a) * len(b)) - 1

def box_with_dots(ax, values, position, color, width=0.35, seed=0):
    values = pd.Series(values).dropna().values
    bp = ax.boxplot([values], positions=[position], widths=width, patch_artist=True,
                    showfliers=False, medianprops=dict(color="#333333"))
    for patch in bp["boxes"]:
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    rng = np.random.default_rng(seed)
    jitter = rng.uniform(-width / 3, width / 3, size=len(values))
    ax.scatter(position + jitter, values, s=9, color=color, alpha=0.7, edgecolors="none", zorder=3)

# ----------------------------- Main steps ----------------------------
def load_data(path):
    df = pd.read_csv(path)
    label_col, id_col = find_column(df, LABEL_NAME), find_column(df, ID_NAME)
    print(f"Loaded {path}: {df.shape[0]} people, {df.shape[1]} columns")
    
    # Ensure labels are lowercased for dictionary matching if they aren't already
    df[label_col] = df[label_col].astype(str).str.lower().str.strip()
    
    labels = sorted(df[label_col].dropna().unique().tolist())
    if len(labels) != 2:
        raise SystemExit(f"Expected 2 classes, found {len(labels)}: {labels}.")
    lo, hi = labels
    names = {lab: LABEL_NAMES.get(lab, str(lab).capitalize()) for lab in labels}
    print(f"\nGroup A (label {lo}) = {names[lo]} | Group B (label {hi}) = {names[hi]}")
    return df, id_col, label_col, lo, hi, names

def parse_features(df, meta_cols):
    rows, skipped = [], []
    for col in df.columns:
        if col in meta_cols:
            continue
        m = FEATURE_PATTERN.match(col)
        if m:
            rows.append(dict(feature=col, band=m.group(1), ch1=m.group(2), ch2=m.group(3)))
        else:
            skipped.append(col)
    info = pd.DataFrame(rows)
    present = [b for b in BAND_ORDER if b in set(info["band"])]
    extra = sorted(set(info["band"]) - set(present))
    info["band"] = pd.Categorical(info["band"], present + extra, ordered=True)
    return info

def feature_table(X, y, lo, hi, names, info, alpha):
    A, B = names[lo], names[hi]
    rows = []
    for col in info["feature"]:
        res = run_mwu(X.loc[y == hi, col], X.loc[y == lo, col])
        if res is None:
            continue
        p, r = res
        med_lo, med_hi = X.loc[y == lo, col].median(), X.loc[y == hi, col].median()
        rows.append({"feature": col, f"median_{A}": med_lo, f"median_{B}": med_hi,
                     "median_diff": med_hi - med_lo, "effect_r": r, "p_value": p})
    t = pd.DataFrame(rows).merge(info, on="feature")
    t["p_fdr"] = false_discovery_control(t["p_value"].values, method="bh")
    t["significant"] = t["p_value"] < alpha
    t["significant_fdr"] = t["p_fdr"] < alpha
    t["higher_in"] = np.where(t["median_diff"] > 0, B, A)
    return t.sort_values("p_value").reset_index(drop=True)

def band_table(X, y, lo, hi, names, info):
    A, B = names[lo], names[hi]
    per_person = {}
    for band, g in info.groupby("band", observed=True):
        per_person[band] = X[g["feature"]].mean(axis=1)
    pp = pd.DataFrame(per_person)
    rows = []
    for band in pp.columns:
        res = run_mwu(pp.loc[y == hi, band], pp.loc[y == lo, band])
        if res is None:
            continue
        p, r = res
        rows.append({"band": band, "n_features": int((info["band"] == band).sum()),
                     f"mean_{A}": pp.loc[y == lo, band].mean(), f"sd_{A}": pp.loc[y == lo, band].std(),
                     f"mean_{B}": pp.loc[y == hi, band].mean(), f"sd_{B}": pp.loc[y == hi, band].std(),
                     "effect_r": r, "p_value": p})
    t = pd.DataFrame(rows)
    t["p_fdr"] = false_discovery_control(t["p_value"].values, method="bh")
    return t, pp

# ----------------------------- Plots ---------------------------------
def plot_band_counts(ft, names, lo, hi, path, alpha):
    A, B = names[lo], names[hi]
    bands = [b for b in ft["band"].cat.categories if (ft["band"] == b).any()]
    sig = ft[ft["significant"]]
    higher_B = [((sig["band"] == b) & (sig["higher_in"] == B)).sum() for b in bands]
    higher_A = [((sig["band"] == b) & (sig["higher_in"] == A)).sum() for b in bands]
    totals = [(ft["band"] == b).sum() for b in bands]
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(bands))
    ax.bar(x, higher_B, color="#C44E52", label=f"Higher in {B}")
    ax.bar(x, higher_A, bottom=higher_B, color="#4C72B0", label=f"Higher in {A}")
    ax.set_xticks(x, [f"{b}\n({t} tested)" for b, t in zip(bands, totals)])
    ax.set_ylabel(f"Number of features with p < {alpha}")
    ax.set_title("Significant features per frequency band")
    ax.legend()
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

def plot_band_boxplots(pp, y, lo, hi, names, bt, path):
    from matplotlib.patches import Patch
    order = list(pp.columns)
    colors = {lo: "#4C72B0", hi: "#C44E52"}
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for i, band in enumerate(order):
        box_with_dots(ax, pp.loc[y == lo, band], i - 0.2, colors[lo], seed=i)
        box_with_dots(ax, pp.loc[y == hi, band], i + 0.2, colors[hi], seed=100 + i)
    top = pp.max().max()
    for i, band in enumerate(order):
        p = bt.loc[bt["band"] == band, "p_value"]
        if len(p):
            ax.text(i, top * 1.02, sig_stars(p.iloc[0]), ha="center", fontsize=12)
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([str(b) for b in order])
    ax.set_ylim(top=top * 1.08)
    ax.set_ylabel("Average wPLI over the band's features")
    ax.set_title("Band-level connectivity by group")
    ax.legend(handles=[Patch(color=colors[lo], alpha=0.6, label=names[lo]),
                       Patch(color=colors[hi], alpha=0.6, label=names[hi])],
              loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

def plot_volcano(ft, path, alpha):
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(data=ft, x="effect_r", y=-np.log10(ft["p_value"]), hue="band",
                    palette="viridis", s=45, alpha=0.8, ax=ax)
    ax.axhline(-np.log10(alpha), color="grey", ls="--", lw=1)
    ax.axvline(0, color="grey", lw=0.8)
    ax.set_xlabel("Effect size (rank-biserial r)   <- higher in A | higher in B ->")
    ax.set_ylabel("-log10 (p-value)")
    ax.set_title("Volcano plot of all features")
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

def plot_top_features(X, y, ft, lo, hi, names, path, n):
    top = ft.head(n)
    ncol = 4
    nrow = int(np.ceil(len(top) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 3.4 * nrow), squeeze=False)
    colors = {lo: "#4C72B0", hi: "#C44E52"}
    for k, (ax, (_, row)) in enumerate(zip(axes.ravel(), top.iterrows())):
        col = row["feature"]
        box_with_dots(ax, X.loc[y == lo, col], 0, colors[lo], width=0.5, seed=k)
        box_with_dots(ax, X.loc[y == hi, col], 1, colors[hi], width=0.5, seed=50 + k)
        ax.set_xticks([0, 1])
        ax.set_xticklabels([names[lo], names[hi]], fontsize=8)
        ax.set_xlim(-0.6, 1.6)
        ax.set_title(f"{col.replace('wPLI_', '')}\np={row['p_value']:.4f}", fontsize=9)
    for ax in axes.ravel()[len(top):]:
        ax.axis("off")
    fig.suptitle(f"Top {len(top)} features by p-value", y=1.0)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

def plot_connectivity_maps(ft, info, path, alpha):
    channels = [c for c in CHANNEL_ORDER if c in set(info["ch1"]) | set(info["ch2"])]
    channels += sorted((set(info["ch1"]) | set(info["ch2"])) - set(channels))
    idx = {c: i for i, c in enumerate(channels)}
    bands = [b for b in ft["band"].cat.categories if (ft["band"] == b).any()]
    vmax = ft.loc[ft["significant"], "median_diff"].abs().max()
    vmax = vmax if np.isfinite(vmax) and vmax > 0 else 1
    cmap = plt.get_cmap("RdBu_r").copy(); cmap.set_bad("#eeeeee")

    ncol = 3
    nrow = int(np.ceil(len(bands) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 5 * nrow), squeeze=False)
    for ax, band in zip(axes.ravel(), bands):
        M = np.full((len(channels), len(channels)), np.nan)
        S = np.zeros_like(M, dtype=bool)
        for _, r in ft[(ft["band"] == band) & ft["significant"]].iterrows():
            i, j = idx[r["ch1"]], idx[r["ch2"]]
            M[i, j] = M[j, i] = r["median_diff"]
            S[i, j] = S[j, i] = r["significant_fdr"]
        im = ax.imshow(np.ma.masked_invalid(M), cmap=cmap, vmin=-vmax, vmax=vmax)
        for i, j in zip(*np.where(S)):
            ax.text(j, i, "*", ha="center", va="center", fontsize=9)
        ax.set_xticks(range(len(channels)), channels, rotation=90, fontsize=7)
        ax.set_yticks(range(len(channels)), channels, fontsize=7)
        n_sig = int(ft[(ft["band"] == band) & ft["significant"]].shape[0])
        ax.set_title(f"{band} ({n_sig} significant pairs)")
        ax.grid(False)
    for ax in axes.ravel()[len(bands):]:
        ax.axis("off")
    cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.7)
    cbar.set_label("Median wPLI difference (group B - group A)")
    fig.suptitle(f"Electrode-pair differences (grey = not significant at p<{alpha}; * = FDR-significant)")
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)

def plot_electrode_counts(ft, path):
    sig = ft[ft["significant"]]
    if sig.empty:
        return
    long = pd.concat([sig[["ch1", "band"]].rename(columns={"ch1": "ch"}),
                      sig[["ch2", "band"]].rename(columns={"ch2": "ch"})])
    tab = long.groupby(["ch", "band"], observed=True).size().unstack(fill_value=0)
    order = [c for c in CHANNEL_ORDER if c in tab.index] + [c for c in tab.index if c not in CHANNEL_ORDER]
    tab = tab.loc[order]
    ax = tab.plot(kind="bar", stacked=True, figsize=(10, 5), colormap="viridis")
    ax.set_xlabel("Electrode"); ax.set_ylabel("Number of significant connections")
    ax.set_title("Which electrodes are involved most?")
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()

def plot_heatmap(X, y, ft, lo, hi, names, path, n):
    cols = ft["feature"].head(n).tolist()
    Z = pd.DataFrame(StandardScaler().fit_transform(X[cols]), columns=[c.replace("wPLI_", "") for c in cols])
    order = np.argsort(y.values, kind="stable")
    colors = y.map({lo: "#4C72B0", hi: "#C44E52"}).values
    g = sns.clustermap(Z.iloc[order], row_cluster=False, col_cluster=True, cmap="vlag", center=0,
                       row_colors=pd.Series(colors[order], name="group").values,
                       figsize=(12, 9), yticklabels=False, xticklabels=True)
    g.ax_heatmap.set_ylabel(f"People (blue = {names[lo]}, red = {names[hi]})")
    g.ax_heatmap.tick_params(axis="x", labelsize=7)
    g.fig.suptitle(f"Top {len(cols)} features (z-scored)", y=1.0)
    g.savefig(path, dpi=150); plt.close(g.fig)

def plot_pca(X, y, info, lo, hi, names, path):
    groups = [("All features", info["feature"].tolist())]
    for band, g in info.groupby("band", observed=True):
        groups.append((f"{band} only", g["feature"].tolist()))
    ncol = 3
    nrow = int(np.ceil(len(groups) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4.5 * nrow), squeeze=False)
    pal = {names[lo]: "#4C72B0", names[hi]: "#C44E52"}
    grp = y.map(names).values
    for ax, (title, cols) in zip(axes.ravel(), groups):
        if len(cols) < 2:
            ax.axis("off"); continue
        pca = PCA(n_components=2)
        P = pca.fit_transform(StandardScaler().fit_transform(X[cols]))
        sns.scatterplot(x=P[:, 0], y=P[:, 1], hue=grp, palette=pal, s=45, ax=ax, legend=(ax is axes[0, 0]))
        ev = pca.explained_variance_ratio_ * 100
        ax.set_xlabel(f"PC1 ({ev[0]:.0f}%)"); ax.set_ylabel(f"PC2 ({ev[1]:.0f}%)")
        ax.set_title(title)
    for ax in axes.ravel()[len(groups):]:
        ax.axis("off")
    fig.suptitle("PCA: do the two groups separate?")
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

# ----------------------------- Run -----------------------------------
def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", default="reduced_features_dataset.csv")
    ap.add_argument("--outdir", default="band_analysis_output")
    ap.add_argument("--alpha", type=float, default=0.05)
    args, _ = ap.parse_known_args(argv)
    out = Path(args.outdir); out.mkdir(parents=True, exist_ok=True)

    df, id_col, label_col, lo, hi, names = load_data(args.input)
    info = parse_features(df, {id_col, label_col})
    X = df[info["feature"]].apply(pd.to_numeric, errors="coerce")
    y = df[label_col]
    print(f"\nFeatures per band:\n{info['band'].value_counts().sort_index().to_string()}")

    ft = feature_table(X, y, lo, hi, names, info, args.alpha)
    bt, pp = band_table(X, y, lo, hi, names, info)
    ft.to_csv(out / "feature_statistics.csv", index=False)
    bt.to_csv(out / "band_statistics.csv", index=False)

    plot_band_counts(ft, names, lo, hi, out / "1_significant_features_per_band.png", args.alpha)
    plot_band_boxplots(pp, y, lo, hi, names, bt, out / "2_band_level_boxplots.png")
    plot_volcano(ft, out / "3_volcano_plot.png", args.alpha)
    plot_top_features(X, y, ft, lo, hi, names, out / "4_top_features_boxplots.png", TOP_N_BOXPLOTS)
    plot_connectivity_maps(ft, info, out / "5_connectivity_difference_maps.png", args.alpha)
    plot_electrode_counts(ft, out / "6_electrode_involvement.png")
    plot_heatmap(X, y, ft, lo, hi, names, out / "7_top_features_heatmap.png", TOP_N_HEATMAP)
    plot_pca(X, y, info, lo, hi, names, out / "8_pca_by_band.png")

    # ---- Console summary ----
    print("\n=== Band-level results (average wPLI per person) ===")
    print(bt.round(4).to_string(index=False))
    print(f"\n=== Features with p < {args.alpha}: {int(ft['significant'].sum())} of {len(ft)} "
          f"| FDR-significant: {int(ft['significant_fdr'].sum())} ===")
    summary = (ft.groupby("band", observed=True)
                 .agg(tested=("feature", "size"), significant=("significant", "sum"),
                      fdr_significant=("significant_fdr", "sum"),
                      median_abs_effect=("effect_r", lambda s: s.abs().median()))
                 .round(3))
    print(summary.to_string())
    print(f"\nTop 10 features:\n{ft[['feature','higher_in','effect_r','p_value','p_fdr']].head(10).round(4).to_string(index=False)}")
    print(f"\nAll tables and figures saved in: {out.resolve()}")

if __name__ == "__main__":
    main()

Loaded reduced_features_dataset.csv: 97 people, 97 columns

Group A (label 0) = 0 | Group B (label 1) = 1

Features per band:
band
Delta     8
Theta     4
Alpha    12
Beta     27
Gamma    44

=== Band-level results (average wPLI per person) ===
 band  n_features  mean_0   sd_0  mean_1   sd_1  effect_r  p_value  p_fdr
Delta           8  0.4386 0.1254  0.3450 0.1127   -0.3838   0.0011 0.0019
Theta           4  0.3566 0.0931  0.3585 0.1197   -0.0579   0.6261 0.6261
Alpha          12  0.2934 0.0708  0.3494 0.0962    0.3609   0.0022 0.0028
 Beta          27  0.2615 0.0495  0.3113 0.0561    0.5438   0.0000 0.0000
Gamma          44  0.3952 0.0927  0.3277 0.0830   -0.4094   0.0005 0.0013

=== Features with p < 0.05: 95 of 95 | FDR-significant: 95 ===
       tested  significant  fdr_significant  median_abs_effect
band                                                          
Delta       8            8                8              0.258
Theta       4            4                4              0

In [14]:
"""
Cross-Band Ratio Analysis: log(Delta/Alpha) and log(Beta/Gamma)

This script calculates regional frequency band ratios to identify biomarkers 
differentiating Schizophrenia from Healthy Controls. 

Statistical suite for each ratio feature:
- Welch's t-test (assumes normal distribution, but robust to unequal variances)
- Mann-Whitney U-test (non-parametric, does not assume normal distribution)
- Cohen's d (parametric effect size)
- Benjamini-Hochberg FDR correction for multiple comparisons
"""

import pandas as pd
import numpy as np
import re
from scipy.stats import ttest_ind, mannwhitneyu
import argparse
from pathlib import Path

# Fallback for older scipy versions missing false_discovery_control
try:
    from scipy.stats import false_discovery_control
except ImportError:
    def false_discovery_control(ps, method="bh"):
        ps = np.asarray(ps)
        order = np.argsort(ps)
        ps_sorted = ps[order]
        n = len(ps)
        q = np.zeros(n)
        prev_q = 1.0
        for i in range(n - 1, -1, -1):
            fdr = min(prev_q, (n / (i + 1)) * ps_sorted[i])
            q[i] = fdr
            prev_q = fdr
        res = np.zeros(n)
        res[order] = q
        return np.clip(res, 0, 1)

# ----------------------------- Settings ------------------------------
LABEL_NAME = "schizophrenia_label"
ID_NAME = "patient id"
EPSILON = 1e-9 # Small constant to prevent log(0) or division by zero errors

def calculate_cohens_d(group1, group2):
    """Calculates Cohen's d for effect size."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    
    # Calculate pooled standard deviation
    pooled_sd = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    # Handle edge case where pooled standard deviation is 0
    if pooled_sd == 0:
        return 0.0
        
    return (np.mean(group1) - np.mean(group2)) / pooled_sd

def find_column(df, wanted):
    clean = lambda s: str(s).strip().lower().replace(" ", "").replace("_", "")
    for col in df.columns:
        if clean(col) == clean(wanted):
            return col
    raise KeyError(f"Could not find a column like '{wanted}'.")

def main(argv=None):
    ap = argparse.ArgumentParser()
    # IMPORTANT: Ensure this is run on the FULL feature dataset, not the reduced one
    ap.add_argument("--input", default="wpli_features (Copy).csv")
    ap.add_argument("--outdir", default="ratio_analysis_output")
    args, _ = ap.parse_known_args(argv)
    
    out = Path(args.outdir)
    out.mkdir(parents=True, exist_ok=True)

    print(f"Loading data from {args.input}...")
    df = pd.read_csv(args.input)
    
    label_col = find_column(df, LABEL_NAME)
    id_col = find_column(df, ID_NAME)
    
    # Ensure labels are lowercased and stripped of spaces
    df[label_col] = df[label_col].astype(str).str.lower().str.strip()
    labels = sorted(df[label_col].dropna().unique().tolist())
    if len(labels) != 2:
        raise SystemExit(f"Expected exactly 2 classes in {label_col}, found: {labels}")
    
    group_a_lbl, group_b_lbl = labels[0], labels[1]
    print(f"Groups identified: Group A = '{group_a_lbl}', Group B = '{group_b_lbl}'")

    # 1. Map features to their electrode pairs
    feature_pattern = re.compile(r"^wPLI_([A-Za-z]+)_([A-Za-z0-9]+-[A-Za-z0-9]+)$")
    pair_dict = {}
    
    for col in df.columns:
        if col in [id_col, label_col]:
            continue
        m = feature_pattern.match(col)
        if m:
            band, pair = m.groups()
            if pair not in pair_dict:
                pair_dict[pair] = {}
            pair_dict[pair][band] = col

    # 2. Calculate Log Ratios
    print("Calculating log(Delta/Alpha) and log(Beta/Gamma) ratios...")
    ratio_df = pd.DataFrame()
    ratio_df[id_col] = df[id_col]
    ratio_df[label_col] = df[label_col]
    
    ratio_features = []
    
    for pair, bands in pair_dict.items():
        # Delta/Alpha Ratio
        if 'Delta' in bands and 'Alpha' in bands:
            feat_name = f"log_Delta_Alpha_{pair}"
            delta_vals = df[bands['Delta']].apply(pd.to_numeric, errors="coerce")
            alpha_vals = df[bands['Alpha']].apply(pd.to_numeric, errors="coerce")
            ratio_df[feat_name] = np.log((delta_vals + EPSILON) / (alpha_vals + EPSILON))
            ratio_features.append(feat_name)
            
        # Beta/Gamma Ratio
        if 'Beta' in bands and 'Gamma' in bands:
            feat_name = f"log_Beta_Gamma_{pair}"
            beta_vals = df[bands['Beta']].apply(pd.to_numeric, errors="coerce")
            gamma_vals = df[bands['Gamma']].apply(pd.to_numeric, errors="coerce")
            ratio_df[feat_name] = np.log((beta_vals + EPSILON) / (gamma_vals + EPSILON))
            ratio_features.append(feat_name)

    print(f"Generated {len(ratio_features)} ratio features.")
    
    # Split datasets
    group_a_df = ratio_df[ratio_df[label_col] == group_a_lbl]
    group_b_df = ratio_df[ratio_df[label_col] == group_b_lbl]

    # 3. Perform Statistical Tests
    print("Running Welch's t-tests, Mann-Whitney U-tests, and calculating Cohen's d...")
    results = []
    
    for feat in ratio_features:
        # Drop NaNs and infinite values created by log bounds
        vals_a = group_a_df[feat].replace([np.inf, -np.inf], np.nan).dropna()
        vals_b = group_b_df[feat].replace([np.inf, -np.inf], np.nan).dropna()
        
        if len(vals_a) == 0 or len(vals_b) == 0:
            continue
            
        # Welch's t-test (equal_var=False)
        t_stat, t_pval = ttest_ind(vals_a, vals_b, equal_var=False, nan_policy='omit')
        
        # Mann-Whitney U-test
        try:
            u_stat, u_pval = mannwhitneyu(vals_a, vals_b, alternative='two-sided')
        except ValueError:
            u_stat, u_pval = np.nan, np.nan
            
        # Cohen's d
        d = calculate_cohens_d(vals_a, vals_b)
        
        results.append({
            "feature": feat,
            "ratio_type": "Delta/Alpha" if "Delta_Alpha" in feat else "Beta/Gamma",
            "electrode_pair": feat.split('_')[-1],
            f"mean_{group_a_lbl}": np.mean(vals_a),
            f"mean_{group_b_lbl}": np.mean(vals_b),
            "cohens_d": d,
            "abs_cohens_d": abs(d),
            "welch_t_stat": t_stat,
            "welch_p_value": t_pval,
            "mwu_u_stat": u_stat,
            "mwu_p_value": u_pval
        })

    results_df = pd.DataFrame(results)
    
    # 4. Apply FDR Correction (Benjamini-Hochberg) to both test types
    results_df["welch_p_fdr"] = false_discovery_control(results_df["welch_p_value"].fillna(1).values, method="bh")
    results_df["mwu_p_fdr"] = false_discovery_control(results_df["mwu_p_value"].fillna(1).values, method="bh")

    # 5. Rank features by absolute Cohen's d (largest effect size first)
    results_df = results_df.sort_values(by="abs_cohens_d", ascending=False).reset_index(drop=True)

    # Export Full Statistical Table
    stats_out_path = out / "ratio_statistics_ranked.csv"
    results_df.to_csv(stats_out_path, index=False)
    
    # Export New Dataset with Ratios
    data_out_path = out / "ratio_features_dataset.csv"
    ratio_df.to_csv(data_out_path, index=False)

    print("\n" + "="*60)
    print("TOP 10 BIOMARKER CANDIDATES (Ranked by Cohen's d)")
    print("="*60)
    # Print the top 10 most impactful features based on effect size
    display_cols = ['feature', 'cohens_d', 'welch_p_value', 'welch_p_fdr', 'mwu_p_value']
    print(results_df[display_cols].head(10).round(5).to_string(index=False))
    
    print("\nData exported to:")
    print(f" - Stats: {stats_out_path}")
    print(f" - Data:  {data_out_path}")

if __name__ == "__main__":
    main()

Loading data from wpli_features (Copy).csv...
Groups identified: Group A = '0', Group B = '1'
Calculating log(Delta/Alpha) and log(Beta/Gamma) ratios...


/tmp/ipykernel_10688/3406709304.py:119: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ratio_df[feat_name] = np.log((delta_vals + EPSILON) / (alpha_vals + EPSILON))
/tmp/ipykernel_10688/3406709304.py:127: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ratio_df[feat_name] = np.log((beta_vals + EPSILON) / (gamma_vals + EPSILON))
/tmp/ipykernel_10688/3406709304.py:119: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all

Generated 342 ratio features.
Running Welch's t-tests, Mann-Whitney U-tests, and calculating Cohen's d...

TOP 10 BIOMARKER CANDIDATES (Ranked by Cohen's d)
              feature  cohens_d  welch_p_value  welch_p_fdr  mwu_p_value
 log_Beta_Gamma_F7-C4  -0.91225        0.00002      0.00694      0.00004
 log_Beta_Gamma_T5-P3  -0.81121        0.00013      0.02142      0.00020
 log_Beta_Gamma_F7-Pz  -0.76978        0.00029      0.03129      0.00052
 log_Beta_Gamma_T4-T5  -0.74566        0.00042      0.03129      0.00070
log_Beta_Gamma_Fp1-Cz  -0.73310        0.00046      0.03129      0.00091
log_Beta_Gamma_Fp2-T3  -0.72328        0.00062      0.03512      0.00063
log_Beta_Gamma_Fp2-T5  -0.67569        0.00127      0.06188      0.00304
 log_Beta_Gamma_T5-Pz  -0.62928        0.00260      0.07897      0.00127
 log_Beta_Gamma_Cz-T5  -0.62231        0.00273      0.07897      0.00118
 log_Beta_Gamma_F8-O2  -0.61966        0.00308      0.07897      0.00312

Data exported to:
 - Stats: ratio_analy

In [19]:
import pandas as pd
import numpy as np

# 1. Load the statistical results and ratio dataset
stats_df = pd.read_csv('ratio_analysis_output/ratio_statistics_ranked.csv')
ratio_df = pd.read_csv('ratio_analysis_output/ratio_features_dataset.csv')

# 2. Set your Cohen's d threshold
threshold = 0.75

# 3. Filter features meeting the threshold
selected_features = stats_df[stats_df['cohens_d'].abs() >= threshold]['feature'].tolist()
available_biomarkers = [col for col in selected_features if col in ratio_df.columns]

print(f"Selected {len(available_biomarkers)} features with |Cohen's d| >= {threshold}")

# 4. Flexibly locate metadata columns (ignoring case, spaces, underscores)
def clean_name(s):
    return str(s).strip().lower().replace(" ", "").replace("_", "")

id_col = None
label_col = None

for col in ratio_df.columns:
    c = clean_name(col)
    if c in ['patientid', 'id', 'subjectid']:
        id_col = col
    elif c in ['schizophrenialabel', 'label', 'group', 'diagnosis']:
        label_col = col

# Fallback: If metadata wasn't included in ratio_features_dataset.csv, recover it from the raw CSV
if id_col is None or label_col is None:
    print("Metadata missing in ratio file; pulling metadata from 'wpli_features (Copy).csv'...")
    raw_df = pd.read_csv('wpli_features (Copy).csv')
    
    for col in raw_df.columns:
        c = clean_name(col)
        if id_col is None and c in ['patientid', 'id', 'subjectid']:
            id_col = col
            ratio_df[id_col] = raw_df[id_col].values
        elif label_col is None and c in ['schizophrenialabel', 'label', 'group', 'diagnosis']:
            label_col = col
            ratio_df[label_col] = raw_df[label_col].values

metadata_cols = [col for col in [id_col, label_col] if col is not None]

# 5. Recombine metadata and features into the final dataset
final_df = ratio_df[metadata_cols + available_biomarkers]

# 6. Save the corrected dataset
output_file = 'final_biomarker_dataset.csv'
final_df.to_csv(output_file, index=False)

print(f"Metadata preserved: {metadata_cols}")
print(f"Saved {final_df.shape[0]} rows and {final_df.shape[1]} columns to '{output_file}'")

Selected 3 features with |Cohen's d| >= 0.75
Metadata preserved: ['Patient_ID', 'Schizophrenia_Label']
Saved 97 rows and 5 columns to 'final_biomarker_dataset.csv'


In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# NVIDIA RAPIDS GPU acceleration
from cuml.preprocessing import StandardScaler
from cuml.linear_model import LogisticRegression

# Validation & evaluation tools
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, precision_recall_curve, auc
)
from sklearn.pipeline import Pipeline

# -----------------------------------------------------------------------------
# 1. Load and Prepare Dataset
# -----------------------------------------------------------------------------
# Adjust file path if needed (e.g., 'final_biomarker_dataset.csv' or 'ratio_features_dataset.csv')
dataset_path = 'final_biomarker_dataset.csv'
df = pd.read_csv(dataset_path)

metadata_keywords = ['patient', 'id', 'label', 'schizophrenia', 'group']
metadata_cols = [col for col in df.columns if any(kw in col.lower() for kw in metadata_keywords)]
feature_cols = [col for col in df.columns if col not in metadata_cols]

label_col = next((col for col in metadata_cols if 'label' in col.lower() or 'schizophrenia' in col.lower()), None)
if not label_col:
    raise ValueError(f"Could not automatically detect the label column. Available columns: {df.columns.tolist()}")

# Fast 32-bit float conversion for GPU memory and compute efficiency
X = df[feature_cols].values.astype(np.float32)

raw_labels = df[label_col].astype(str)
y = raw_labels.map({'0': 0, '1': 1, 'healthy': 0, 'schizophrenia': 1, 'Healthy': 0, 'Schizophrenia': 1}).values.astype(np.int32)

print(f"Dataset Loaded: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Class Balance: Healthy (0) = {np.sum(y == 0)} | Schizophrenia (1) = {np.sum(y == 1)}")

# -----------------------------------------------------------------------------
# 2. Configure Rigorous Cross-Validation Pipeline
# -----------------------------------------------------------------------------
n_splits = 5
n_repeats = 10
cv = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=42)

# Metrics tracking across folds
train_aucs, test_aucs = [], []
train_accs, test_accs = [], []
test_precisions, test_recalls, test_f1s = [], [], []
test_specificities = []

# Out-of-fold probability accumulator for overall ROC/PR curves
oof_probs = np.zeros((len(y), n_repeats))
oof_counts = np.zeros((len(y), n_repeats))

# Feature coefficient tracker across all folds
coef_list = []

print(f"\nRunning Repeated Stratified Cross-Validation ({n_repeats} repeats x {n_splits} folds = {n_repeats * n_splits} fits)...")

fold_counter = 0
for repeat_idx in range(n_repeats):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42 + repeat_idx)
    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Pipeline instantiation (GPU acceleration via cuML)
        scaler = StandardScaler()
        model = LogisticRegression(penalty='l2', C=1.0, solver='qn', max_iter=1000)

        # Fit on training fold
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        model.fit(X_train_scaled, y_train)

        # Predictions & Probabilities
        train_pred_prob = model.predict_proba(X_train_scaled)[:, 1]
        val_pred_prob = model.predict_proba(X_val_scaled)[:, 1]
        val_pred = (val_pred_prob >= 0.5).astype(int)
        train_pred = (train_pred_prob >= 0.5).astype(int)

        # Accumulate out-of-fold probabilities
        oof_probs[val_idx, repeat_idx] = val_pred_prob

        # Fold metrics
        train_aucs.append(roc_auc_score(y_train, train_pred_prob))
        test_aucs.append(roc_auc_score(y_val, val_pred_prob))
        train_accs.append(accuracy_score(y_train, train_pred))
        test_accs.append(accuracy_score(y_val, val_pred))

        test_precisions.append(precision_score(y_val, val_pred, zero_division=0))
        test_recalls.append(recall_score(y_val, val_pred, zero_division=0))  # Sensitivity
        test_f1s.append(f1_score(y_val, val_pred, zero_division=0))

        # Specificity = TN / (TN + FP)
        cm_fold = confusion_matrix(y_val, val_pred, labels=[0, 1])
        tn, fp, fn, tp = cm_fold.ravel() if cm_fold.shape == (2, 2) else (0, 0, 0, 0)
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        test_specificities.append(spec)

        # Store model coefficients (convert GPU array to CPU numpy)
        coef_list.append(model.coef_.to_numpy() if hasattr(model.coef_, 'to_numpy') else model.coef_)

        fold_counter += 1

# Average out-of-fold predictions across repeats
mean_oof_probs = oof_probs.mean(axis=1)
mean_oof_preds = (mean_oof_probs >= 0.5).astype(int)

# -----------------------------------------------------------------------------
# 3. Model Diagnostic Report & Overfitting Check
# -----------------------------------------------------------------------------
train_auc_mean, train_auc_std = np.mean(train_aucs), np.std(train_aucs)
test_auc_mean, test_auc_std = np.mean(test_aucs), np.std(test_aucs)
auc_gap = train_auc_mean - test_auc_mean

print("\n" + "=" * 60)
print("              MODEL VALIDATION & PERFORMANCE METRICS          ")
print("=" * 60)
print(f"ROC-AUC (Validation) : {test_auc_mean:.4f} ± {test_auc_std:.4f}")
print(f"ROC-AUC (Training)   : {train_auc_mean:.4f} ± {train_auc_std:.4f}")
print(f"Accuracy (Validation): {np.mean(test_accs):.4f} ± {np.std(test_accs):.4f}")
print(f"Accuracy (Training)  : {np.mean(train_accs):.4f} ± {np.std(train_accs):.4f}")
print("-" * 60)
print(f"Sensitivity (Recall) : {np.mean(test_recalls):.4f} ± {np.std(test_recalls):.4f}")
print(f"Specificity          : {np.mean(test_specificities):.4f} ± {np.std(test_specificities):.4f}")
print(f"Precision            : {np.mean(test_precisions):.4f} ± {np.std(test_precisions):.4f}")
print(f"F1-Score             : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")
print("=" * 60)

print(f"\n[Overfitting Assessment] Generalization Gap (Train AUC - Val AUC): {auc_gap:.4f}")
if auc_gap > 0.08:
    print("-> HIGH OVERFITTING RISK: Train AUC is substantially higher than Val AUC.")
    print("   Recommendation: Increase L2 penalty (use lower C, e.g., C=0.1 or C=0.01) or reduce feature count.")
elif auc_gap > 0.04:
    print("-> MILD OVERFITTING: Slight performance drop on validation folds, but within acceptable limits.")
else:
    print("-> EXCELLENT GENERALIZATION: Minimal gap between training and validation performance.")

# -----------------------------------------------------------------------------
# 4. Generate Publication-Quality Visualizations
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
sns.set_theme(style="whitegrid", context="notebook")

# A. ROC Curve (Out-of-Fold)
fpr, tpr, _ = roc_curve(y, mean_oof_probs)
oof_auc = auc(fpr, tpr)
axes[0, 0].plot(fpr, tpr, color="#C44E52", lw=2, label=f"OOF ROC (AUC = {oof_auc:.3f})")
axes[0, 0].plot([0, 1], [0, 1], color="grey", linestyle="--", lw=1)
axes[0, 0].set_xlabel("False Positive Rate (1 - Specificity)")
axes[0, 0].set_ylabel("True Positive Rate (Sensitivity)")
axes[0, 0].set_title("A. Out-of-Fold Receiver Operating Characteristic")
axes[0, 0].legend(loc="lower right")

# B. Normalized Confusion Matrix
cm_overall = confusion_matrix(y, mean_oof_preds, normalize='true')
sns.heatmap(cm_overall, annot=True, fmt=".2%", cmap="Blues", ax=axes[0, 1],
            xticklabels=["Healthy", "Schizophrenia"], yticklabels=["Healthy", "Schizophrenia"],
            cbar=False, annot_kws={"size": 14})
axes[0, 1].set_xlabel("Predicted Label")
axes[0, 1].set_ylabel("True Label")
axes[0, 1].set_title("B. Overall Normalized Confusion Matrix")

# C. Precision-Recall Curve
prec_array, rec_array, _ = precision_recall_curve(y, mean_oof_probs)
pr_auc = auc(rec_array, prec_array)
axes[1, 0].plot(rec_array, prec_array, color="#4C72B0", lw=2, label=f"PR Curve (AUC = {pr_auc:.3f})")
axes[1, 0].set_xlabel("Recall (Sensitivity)")
axes[1, 0].set_ylabel("Precision")
axes[1, 0].set_title("C. Precision-Recall Curve")
axes[1, 0].legend(loc="lower left")

# D. Feature Coefficients & Stability Across Folds
mean_coefs = np.mean(np.vstack(coef_list), axis=0).flatten()
std_coefs = np.std(np.vstack(coef_list), axis=0).flatten()

coef_df = pd.DataFrame({
    'Feature': [f.replace('log_', '') for f in feature_cols],
    'Mean_Coef': mean_coefs,
    'Std_Coef': std_coefs
}).sort_values(by='Mean_Coef', key=abs, ascending=True)

y_pos = np.arange(len(coef_df))
axes[1, 1].barh(y_pos, coef_df['Mean_Coef'], xerr=coef_df['Std_Coef'], align='center',
                 color=np.where(coef_df['Mean_Coef'] > 0, '#C44E52', '#4C72B0'),
                 alpha=0.8, ecolor='black', capsize=3)
axes[1, 1].set_yticks(y_pos)
axes[1, 1].set_yticklabels(coef_df['Feature'], fontsize=9)
axes[1, 1].axvline(0, color='grey', lw=0.8)
axes[1, 1].set_xlabel("L2 Model Weight (Log-Odds)")
axes[1, 1].set_title("D. Feature Importance & Stability Across Folds")

plt.tight_layout()
output_fig = 'model_validation_dashboard.png'
plt.savefig(output_fig, dpi=150)
print(f"\nComprehensive diagnostic dashboard saved as '{output_fig}'")

Dataset Loaded: 97 samples, 3 features
Class Balance: Healthy (0) = 50 | Schizophrenia (1) = 47

Running Repeated Stratified Cross-Validation (10 repeats x 5 folds = 50 fits)...

              MODEL VALIDATION & PERFORMANCE METRICS          
ROC-AUC (Validation) : 0.7858 ± 0.0886
ROC-AUC (Training)   : 0.8025 ± 0.0224
Accuracy (Validation): 0.7003 ± 0.0920
Accuracy (Training)  : 0.7167 ± 0.0297
------------------------------------------------------------
Sensitivity (Recall) : 0.6811 ± 0.1441
Specificity          : 0.7180 ± 0.1410
Precision            : 0.7046 ± 0.1149
F1-Score             : 0.6833 ± 0.1069

[Overfitting Assessment] Generalization Gap (Train AUC - Val AUC): 0.0167
-> EXCELLENT GENERALIZATION: Minimal gap between training and validation performance.

Comprehensive diagnostic dashboard saved as 'model_validation_dashboard.png'


In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

# NVIDIA RAPIDS GPU acceleration
from cuml.preprocessing import StandardScaler
from cuml.linear_model import LogisticRegression

# Validation & evaluation tools
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, precision_recall_curve, auc
)

# -----------------------------------------------------------------------------
# 1. Load and Prepare Dataset
# -----------------------------------------------------------------------------
df = pd.read_csv('final_biomarker_dataset.csv')

metadata_keywords = ['patient', 'id', 'label', 'schizophrenia', 'group']
metadata_cols = [col for col in df.columns if any(kw in col.lower() for kw in metadata_keywords)]
feature_cols = [col for col in df.columns if col not in metadata_cols]

label_col = next((col for col in metadata_cols if 'label' in col.lower() or 'schizophrenia' in col.lower()), None)

# Fast 32-bit float conversion for GPU memory and compute efficiency
X = df[feature_cols].values.astype(np.float32)
raw_labels = df[label_col].astype(str)
y = raw_labels.map({'0': 0, '1': 1, 'healthy': 0, 'schizophrenia': 1, 'Healthy': 0, 'Schizophrenia': 1}).values.astype(np.int32)

# -----------------------------------------------------------------------------
# 2. Configure Rigorous Cross-Validation Pipeline
# -----------------------------------------------------------------------------
n_splits = 5
n_repeats = 10

train_aucs, test_aucs = [], []
train_accs, test_accs = [], []
coef_list = []

# Out-of-fold probability accumulator 
oof_probs = np.zeros((len(y), n_repeats))

print(f"Running Rigorous Cross-Validation ({n_repeats} repeats x {n_splits} folds = {n_repeats * n_splits} GPU fits)...")

for repeat_idx in range(n_repeats):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42 + repeat_idx)
    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Pipeline instantiation (GPU acceleration)
        scaler = StandardScaler()
        model = LogisticRegression(penalty='l2', C=1.0, solver='qn', max_iter=1000)

        # Fit on training fold
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        model.fit(X_train_scaled, y_train)

        # Predictions & Probabilities
        train_pred_prob = model.predict_proba(X_train_scaled)[:, 1]
        val_pred_prob = model.predict_proba(X_val_scaled)[:, 1]
        
        # Accumulate out-of-fold probabilities
        oof_probs[val_idx, repeat_idx] = val_pred_prob

        # Track Overfitting Metrics
        train_aucs.append(roc_auc_score(y_train, train_pred_prob))
        test_aucs.append(roc_auc_score(y_val, val_pred_prob))
        
        # Store model coefficients 
        coef_list.append(model.coef_.to_numpy() if hasattr(model.coef_, 'to_numpy') else model.coef_)

# Average out-of-fold predictions across repeats
mean_oof_probs = oof_probs.mean(axis=1)

# -----------------------------------------------------------------------------
# 3. Calculate Optimal Threshold & Final Metrics
# -----------------------------------------------------------------------------
fpr, tpr, thresholds = roc_curve(y, mean_oof_probs)
specificities = 1 - fpr
sensitivities = tpr

# Youden's J statistic maximizes the sum of sensitivity and specificity
youden_j = sensitivities + specificities - 1
optimal_idx = np.argmax(youden_j)
optimal_threshold = thresholds[optimal_idx]

# Apply optimal threshold to get definitive binary predictions
optimized_preds = (mean_oof_probs >= optimal_threshold).astype(int)

# -----------------------------------------------------------------------------
# 4. Model Diagnostic Report & Overfitting Check
# -----------------------------------------------------------------------------
train_auc_mean, train_auc_std = np.mean(train_aucs), np.std(train_aucs)
test_auc_mean, test_auc_std = np.mean(test_aucs), np.std(test_aucs)
auc_gap = train_auc_mean - test_auc_mean

print("\n" + "=" * 60)
print("              MODEL VALIDATION & PERFORMANCE METRICS          ")
print("=" * 60)
print(f"ROC-AUC (Validation) : {test_auc_mean:.4f} ± {test_auc_std:.4f}")
print(f"ROC-AUC (Training)   : {train_auc_mean:.4f} ± {train_auc_std:.4f}")
print("-" * 60)
print(f"Decision Threshold   : Optimized cutoff at p >= {optimal_threshold:.4f}")
print(f"Sensitivity (Recall) : {recall_score(y, optimized_preds):.4f}")
print(f"Specificity          : {specificities[optimal_idx]:.4f}")
print(f"Precision            : {precision_score(y, optimized_preds):.4f}")
print(f"F1-Score             : {f1_score(y, optimized_preds):.4f}")
print("=" * 60)

print(f"\n[Overfitting Assessment] Generalization Gap (Train AUC - Val AUC): {auc_gap:.4f}")
if auc_gap > 0.08:
    print("-> HIGH OVERFITTING RISK: Train AUC is substantially higher than Val AUC.")
elif auc_gap > 0.04:
    print("-> MILD OVERFITTING: Slight performance drop on validation folds, but within acceptable limits.")
else:
    print("-> EXCELLENT GENERALIZATION: Minimal gap between training and validation performance.")

# -----------------------------------------------------------------------------
# 5. Train Final Deployable Model
# -----------------------------------------------------------------------------
print("\nTraining Final Deployable Model on 100% of the dataset...")
final_scaler = StandardScaler()
final_model = LogisticRegression(penalty='l2', C=1.0, solver='qn', max_iter=1000)

X_scaled_final = final_scaler.fit_transform(X)
final_model.fit(X_scaled_final, y)

model_package = {
    'scaler': final_scaler,
    'model': final_model,
    'optimal_threshold': optimal_threshold,
    'features': feature_cols
}

output_model_file = 'optimized_schizophrenia_model.pkl'
with open(output_model_file, 'wb') as f:
    pickle.dump(model_package, f)
print(f"Final Model successfully exported to: {output_model_file}")

# -----------------------------------------------------------------------------
# 6. Generate Diagnostics Dashboard
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
sns.set_theme(style="whitegrid", context="notebook")

# A. ROC Curve 
oof_auc = auc(fpr, tpr)
axes[0, 0].plot(fpr, tpr, color="#C44E52", lw=2, label=f"OOF ROC (AUC = {oof_auc:.3f})")
axes[0, 0].plot([0, 1], [0, 1], color="grey", linestyle="--", lw=1)
axes[0, 0].scatter(fpr[optimal_idx], tpr[optimal_idx], color='black', s=80, zorder=5, label=f'Optimal Cutoff (p={optimal_threshold:.2f})')
axes[0, 0].set_xlabel("False Positive Rate (1 - Specificity)")
axes[0, 0].set_ylabel("True Positive Rate (Sensitivity)")
axes[0, 0].set_title("A. Out-of-Fold ROC Curve")
axes[0, 0].legend(loc="lower right")

# B. Sensitivity vs Specificity Tradeoff
axes[0, 1].plot(thresholds, sensitivities, label='Sensitivity', color='#C44E52', lw=2)
axes[0, 1].plot(thresholds, specificities, label='Specificity', color='#4C72B0', lw=2)
axes[0, 1].axvline(optimal_threshold, color='k', linestyle='--', lw=1.5, label='Optimal Threshold')
axes[0, 1].set_xlim([0.0, 1.0])
axes[0, 1].set_xlabel('Probability Decision Threshold')
axes[0, 1].set_ylabel('Score')
axes[0, 1].set_title("B. Threshold Optimization Tradeoff")
axes[0, 1].legend(loc='lower center')

# C. Optimized Confusion Matrix
cm_overall = confusion_matrix(y, optimized_preds, normalize='true')
sns.heatmap(cm_overall, annot=True, fmt=".2%", cmap="Blues", ax=axes[1, 0],
            xticklabels=["Healthy", "Schizophrenia"], yticklabels=["Healthy", "Schizophrenia"],
            cbar=False, annot_kws={"size": 14})
axes[1, 0].set_xlabel("Predicted Label")
axes[1, 0].set_ylabel("True Label")
axes[1, 0].set_title(f"C. Confusion Matrix (Threshold = {optimal_threshold:.2f})")

# D. Feature Coefficients Stability
mean_coefs = np.mean(np.vstack(coef_list), axis=0).flatten()
std_coefs = np.std(np.vstack(coef_list), axis=0).flatten()

coef_df = pd.DataFrame({
    'Feature': [f.replace('log_', '') for f in feature_cols],
    'Mean_Coef': mean_coefs,
    'Std_Coef': std_coefs
}).sort_values(by='Mean_Coef', key=abs, ascending=True)

y_pos = np.arange(len(coef_df))
axes[1, 1].barh(y_pos, coef_df['Mean_Coef'], xerr=coef_df['Std_Coef'], align='center',
                 color=np.where(coef_df['Mean_Coef'] > 0, '#C44E52', '#4C72B0'),
                 alpha=0.8, ecolor='black', capsize=3)
axes[1, 1].set_yticks(y_pos)
axes[1, 1].set_yticklabels(coef_df['Feature'], fontsize=9)
axes[1, 1].axvline(0, color='grey', lw=0.8)
axes[1, 1].set_xlabel("L2 Model Weight (Log-Odds)")
axes[1, 1].set_title("D. Feature Importance & Stability Across 50 Folds")

plt.tight_layout()
output_fig = 'comprehensive_model_dashboard.png'
plt.savefig(output_fig, dpi=150)


Running Rigorous Cross-Validation (10 repeats x 5 folds = 50 GPU fits)...

              MODEL VALIDATION & PERFORMANCE METRICS          
ROC-AUC (Validation) : 0.7858 ± 0.0886
ROC-AUC (Training)   : 0.8025 ± 0.0224
------------------------------------------------------------
Decision Threshold   : Optimized cutoff at p >= 0.3891
Sensitivity (Recall) : 0.8936
Specificity          : 0.6200
Precision            : 0.6885
F1-Score             : 0.7778

[Overfitting Assessment] Generalization Gap (Train AUC - Val AUC): 0.0167
-> EXCELLENT GENERALIZATION: Minimal gap between training and validation performance.

Training Final Deployable Model on 100% of the dataset...
Final Model successfully exported to: optimized_schizophrenia_model.pkl

Comprehensive diagnostic dashboard saved as 'comprehensive_model_dashboard.png'
